# Ragged Edge Simulation

**Module 03 — Notebook 02**  
**Estimated time:** 15 minutes

## Learning Objectives

1. Simulate MIDAS data with known true parameters to measure ragged-edge estimation bias
2. Compare the three ragged-edge handling strategies (backward shift, re-weighting, no-correction)
3. Quantify how much the nowcast degrades at the 1-month vs. 3-month vintage
4. Show that front-loaded weight functions are more affected by the ragged edge

## Key Idea

Simulation lets us know the true parameters and measure how well each strategy recovers them. In real data, we can only compare OOS RMSE — in simulation, we can also measure parameter bias directly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded.")

## 1. Simulation Setup

We generate synthetic MIDAS data with known true parameters:

$$y_t = \alpha_0 + \beta_0 \sum_{j=0}^{K-1} w_j(\theta_1^0, \theta_2^0) x_{mt-j} + \varepsilon_t$$

Then apply MIDAS estimation at each vintage point (h=0,1,2) and compare recovered parameters to true values.

In [ ]:
def beta_weights(K, theta1, theta2):
    if theta1 <= 0 or theta2 <= 0:
        return np.ones(K) / K
    x = (np.arange(K) + 0.5) / K
    raw = beta_dist.pdf(1 - x, theta1, theta2)
    s = raw.sum()
    return raw / s if s > 1e-12 else np.ones(K) / K


def estimate_midas_from_matrix(Y, X, starts=None):
    """Profile NLS for arbitrary K (handles ragged edge X)."""
    K = X.shape[1]
    if starts is None:
        starts = [(1.0, 5.0), (1.5, 4.0), (2.0, 3.0)]

    def psse(theta):
        t1, t2 = theta
        if t1 <= 0.01 or t2 <= 0.01: return 1e10
        x_pts = (np.arange(K) + 0.5) / K
        raw = beta_dist.pdf(1 - x_pts, t1, t2)
        s = raw.sum()
        w = raw / s if s > 1e-12 else np.ones(K) / K
        xw = X @ w
        xc, yc = xw - xw.mean(), Y - Y.mean()
        ss = np.dot(xc, xc)
        if ss < 1e-12: return np.sum((Y - Y.mean())**2)
        b = np.dot(yc, xc) / ss
        a = Y.mean() - b * xw.mean()
        return np.sum((Y - a - b * xw)**2)

    best_sse, best_res = np.inf, None
    for t0 in starts:
        res = minimize(psse, t0, method='Nelder-Mead',
                       options={'maxiter': 20000, 'xatol': 1e-8})
        if res.fun < best_sse:
            best_sse, best_res = res.fun, res

    t1, t2 = max(best_res.x[0], 0.01), max(best_res.x[1], 0.01)
    K_eff = X.shape[1]
    x_pts = (np.arange(K_eff) + 0.5) / K_eff
    raw = beta_dist.pdf(1 - x_pts, t1, t2)
    s = raw.sum()
    w = raw / s if s > 1e-12 else np.ones(K_eff) / K_eff
    xw = X @ w
    xc, yc = xw - xw.mean(), Y - Y.mean()
    beta_ = np.dot(yc, xc) / np.dot(xc, xc)
    alpha_ = Y.mean() - beta_ * xw.mean()
    return {'theta1': t1, 'theta2': t2, 'alpha': alpha_, 'beta': beta_, 'weights': w}


def simulate_midas(T, K, theta1_true, theta2_true, alpha_true, beta_true,
                   sigma_eps=0.5, sigma_x=1.0, seed=None):
    """
    Generate synthetic MIDAS data with known true parameters.

    Returns
    -------
    Y : np.ndarray (T,)
    X : np.ndarray (T, K)  — complete-quarter data matrix
    w_true : np.ndarray (K,)
    """
    if seed is not None:
        np.random.seed(seed)

    # Monthly HF data: AR(1) with mild autocorrelation
    n_monthly = T * 3 + K
    x_monthly = np.zeros(n_monthly)
    for t in range(1, n_monthly):
        x_monthly[t] = 0.3 * x_monthly[t-1] + sigma_x * np.random.randn()

    # Build MIDAS matrix: T rows, K monthly lags
    X = np.zeros((T, K))
    for t in range(T):
        end_idx = K + t * 3  # 3 months per quarter
        for j in range(K):
            X[t, j] = x_monthly[end_idx - 1 - j]

    # True weight function
    w_true = beta_weights(K, theta1_true, theta2_true)

    # Generate Y
    xw_true = X @ w_true
    eps = sigma_eps * np.random.randn(T)
    Y = alpha_true + beta_true * xw_true + eps

    return Y, X, w_true


# True parameters
TRUE_THETA1 = 1.4
TRUE_THETA2 = 5.0
TRUE_ALPHA  = 0.4
TRUE_BETA   = 0.6
T_SIM  = 100
K_SIM  = 12
SIGMA  = 0.6

Y_sim, X_sim, w_true = simulate_midas(
    T_SIM, K_SIM, TRUE_THETA1, TRUE_THETA2, TRUE_ALPHA, TRUE_BETA,
    sigma_eps=SIGMA, seed=42
)

print(f"Simulated MIDAS data: T={T_SIM}, K={K_SIM}")
print(f"True parameters: alpha={TRUE_ALPHA}, beta={TRUE_BETA}, theta=({TRUE_THETA1}, {TRUE_THETA2})")
print(f"True weight on j=0 (most recent): {w_true[0]:.4f} vs equal weight {1/K_SIM:.4f}")
print(f"True weight concentration in first 3 lags: {w_true[:3].sum():.4f} vs equal 0.25")

## 2. Ragged Edge Strategies

Three strategies for the ragged-edge problem when `h_missing` months are unavailable:

1. **Backward shift (truncate):** Use only K - h_missing lags, drop the front
2. **Re-weighting (correct):** Use available lags with weights renormalized to sum to 1
3. **No correction (naive):** Treat missing lags as zero

In [ ]:
def apply_ragged_edge(X_full, h_missing, strategy='truncate'):
    """
    Apply a ragged-edge strategy to create a nowcast data matrix.

    Parameters
    ----------
    X_full : np.ndarray (T, K) — complete-quarter data
    h_missing : int — number of missing most-recent lags
    strategy : 'truncate', 'reweight', or 'zero_fill'

    Returns
    -------
    X_ragged : np.ndarray
    """
    K = X_full.shape[1]
    if h_missing == 0:
        return X_full

    if strategy == 'truncate':
        # Drop the h_missing most recent lags
        return X_full[:, h_missing:]  # Shape (T, K - h_missing)

    elif strategy == 'reweight':
        # Keep all K columns but zero-fill missing lags
        # (the re-weighting happens at prediction time)
        X_rag = X_full.copy()
        X_rag[:, :h_missing] = 0.0
        return X_rag  # Shape (T, K) with leading zeros

    elif strategy == 'zero_fill':
        # Naive: treat missing as zero (no correction)
        X_rag = X_full.copy()
        X_rag[:, :h_missing] = 0.0
        return X_rag  # Shape (T, K)

    else:
        raise ValueError(f"Unknown strategy: {strategy}")


def nowcast_reweight(alpha, beta, theta1, theta2, x_avail, h_missing, K):
    """
    Ragged-edge nowcast with re-weighting:
    Use available lags h_missing...K-1 with weights renormalized.
    """
    w_full = beta_weights(K, theta1, theta2)
    w_avail = w_full[h_missing:]  # Weights for available lags
    w_norm = w_avail / w_avail.sum()  # Renormalize
    # Scale by the total available mass to maintain scale
    xw = x_avail @ w_norm * w_avail.sum()
    return alpha + beta * xw


print("Strategy functions defined.")
print(f"Backward shift at h=1: X shape {apply_ragged_edge(X_sim, 1, 'truncate').shape}")
print(f"Re-weight at h=1: X shape {apply_ragged_edge(X_sim, 1, 'reweight').shape}")

## 3. Single-Sample Bias Analysis

Estimate MIDAS on the simulated data at each vintage point and compare recovered parameters to true values.

In [ ]:
print("Single-sample estimation by vintage and strategy...")
print("=" * 70)
print(f"True: alpha={TRUE_ALPHA:.4f}, beta={TRUE_BETA:.4f}, theta=({TRUE_THETA1:.4f}, {TRUE_THETA2:.4f})")
print()

strategies = [('truncate', 'Backward shift'), ('reweight', 'Re-weighting'), ('zero_fill', 'Zero-fill (naive)')]
single_sample_results = {}

print(f"{'h_miss':<8} {'Strategy':<20} {'alpha':>8} {'beta':>8} {'theta1':>8} {'theta2':>8} {'RMSE':>8}")
print("-" * 70)

# Complete-quarter baseline
est0 = estimate_midas_from_matrix(Y_sim, X_sim)
resid0 = Y_sim - est0['alpha'] - est0['beta'] * (X_sim @ est0['weights'])
rmse0 = np.sqrt(np.mean(resid0**2))
print(f"{'h=0':<8} {'Complete quarter':<20} "
      f"{est0['alpha']:>8.4f} {est0['beta']:>8.4f} "
      f"{est0['theta1']:>8.4f} {est0['theta2']:>8.4f} {rmse0:>8.4f}")
single_sample_results[(0, 'complete')] = est0

for h in [1, 2]:
    for strat_key, strat_name in strategies:
        X_rag = apply_ragged_edge(X_sim, h, strat_key)
        est = estimate_midas_from_matrix(Y_sim, X_rag)

        # Compute predicted values (use available lags)
        xw_hat = X_rag @ est['weights']
        resid = Y_sim - est['alpha'] - est['beta'] * xw_hat
        rmse = np.sqrt(np.mean(resid**2))

        print(f"{'h='+str(h):<8} {strat_name:<20} "
              f"{est['alpha']:>8.4f} {est['beta']:>8.4f} "
              f"{est['theta1']:>8.4f} {est['theta2']:>8.4f} {rmse:>8.4f}")
        single_sample_results[(h, strat_key)] = est

    print()

print("-" * 70)
print(f"{'True':<8} {'':20} "
      f"{TRUE_ALPHA:>8.4f} {TRUE_BETA:>8.4f} "
      f"{TRUE_THETA1:>8.4f} {TRUE_THETA2:>8.4f}")

## 4. Monte Carlo: Bias and Variance by Vintage

Run 100 Monte Carlo replications to measure the distribution of estimators at each vintage point.

In [ ]:
N_MC = 100  # Monte Carlo replications
T_MC = 80
K_MC = 12

print(f"Running {N_MC} Monte Carlo replications (T={T_MC}, K={K_MC})...")

# Storage: [beta, theta1, theta2] for each h and strategy
mc_results = {
    (h, strat): {'beta': [], 'theta1': [], 'theta2': []}
    for h in [0, 1, 2]
    for strat in ['truncate', 'reweight']
}

for rep in range(N_MC):
    Y_mc, X_mc, _ = simulate_midas(
        T_MC, K_MC, TRUE_THETA1, TRUE_THETA2, TRUE_ALPHA, TRUE_BETA,
        sigma_eps=SIGMA, seed=rep
    )

    for h in [0, 1, 2]:
        for strat in ['truncate', 'reweight']:
            X_rag = apply_ragged_edge(X_mc, h, strat)
            try:
                est = estimate_midas_from_matrix(Y_mc, X_rag, starts=[(1.0, 5.0), (1.5, 4.0)])
                mc_results[(h, strat)]['beta'].append(est['beta'])
                mc_results[(h, strat)]['theta1'].append(est['theta1'])
                mc_results[(h, strat)]['theta2'].append(est['theta2'])
            except Exception:
                pass

    if (rep + 1) % 25 == 0:
        print(f"  Rep {rep+1}/{N_MC}")

print("Monte Carlo complete.")

In [ ]:
# --- Summarize MC results ---
print("Monte Carlo Results: Bias and RMSE by Vintage and Strategy")
print("=" * 75)
print(f"{'h':<5} {'Strategy':<18} {'beta bias':>10} {'beta RMSE':>10} {'θ1 bias':>10} {'θ2 bias':>10}")
print("-" * 75)

mc_summary = {}
for h in [0, 1, 2]:
    for strat, label in [('truncate', 'Backward shift'), ('reweight', 'Re-weighting')]:
        data = mc_results[(h, strat)]
        if len(data['beta']) == 0:
            continue
        betas = np.array(data['beta'])
        t1s = np.array(data['theta1'])
        t2s = np.array(data['theta2'])

        beta_bias = betas.mean() - TRUE_BETA
        beta_rmse = np.sqrt(np.mean((betas - TRUE_BETA)**2))
        t1_bias = t1s.mean() - TRUE_THETA1
        t2_bias = t2s.mean() - TRUE_THETA2

        mc_summary[(h, strat)] = {
            'beta_bias': beta_bias, 'beta_rmse': beta_rmse,
            't1_bias': t1_bias, 't2_bias': t2_bias,
            'betas': betas, 't2s': t2s
        }
        print(f"h={h:<4} {label:<18} {beta_bias:>+10.4f} {beta_rmse:>10.4f} "
              f"{t1_bias:>+10.4f} {t2_bias:>+10.4f}")

print("-" * 75)
print(f"True: beta={TRUE_BETA}, theta1={TRUE_THETA1}, theta2={TRUE_THETA2}")

In [ ]:
# --- Visualization: MC distributions by vintage ---
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, h in enumerate([0, 1, 2]):
    vintage_name = {0: '3-month\n(complete)', 1: '2-month\n(h=1)', 2: '1-month\n(h=2)'}[h]

    # Row 0: beta distribution
    ax = axes[0, col]
    for strat, label, color in [
        ('truncate', 'Backward shift', 'steelblue'),
        ('reweight', 'Re-weighting', 'tomato')
    ]:
        if (h, strat) in mc_summary:
            betas = mc_summary[(h, strat)]['betas']
            ax.hist(betas, bins=20, alpha=0.6, color=color, edgecolor='white', label=label)

    ax.axvline(TRUE_BETA, color='black', linewidth=2, linestyle='-', label=f'True β={TRUE_BETA}')
    ax.set_title(f'β distribution — {vintage_name}')
    ax.set_xlabel('Estimated β')
    ax.set_ylabel('Count')
    if col == 0:
        ax.legend(fontsize=8)

    # Row 1: theta2 distribution
    ax2 = axes[1, col]
    for strat, label, color in [
        ('truncate', 'Backward shift', 'steelblue'),
        ('reweight', 'Re-weighting', 'tomato')
    ]:
        if (h, strat) in mc_summary:
            t2s = mc_summary[(h, strat)]['t2s']
            ax2.hist(t2s, bins=20, alpha=0.6, color=color, edgecolor='white', label=label)

    ax2.axvline(TRUE_THETA2, color='black', linewidth=2, linestyle='-',
                label=f'True θ₂={TRUE_THETA2}')
    ax2.set_title(f'θ₂ distribution — {vintage_name}')
    ax2.set_xlabel('Estimated θ₂')
    ax2.set_ylabel('Count')
    if col == 0:
        ax2.legend(fontsize=8)

plt.suptitle(f'Monte Carlo: Ragged Edge Effect on MIDAS Estimation\n'
             f'(N={N_MC}, T={T_MC}, K={K_MC}, True θ=({TRUE_THETA1},{TRUE_THETA2}))',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('ragged_edge_mc.png', dpi=120, bbox_inches='tight')
plt.show()
print("Figure saved.")

## 5. Front-Loaded vs. Flat Weights: Ragged Edge Sensitivity

In [ ]:
# Show how the ragged edge effect depends on how front-loaded the weights are

# Different true weight shapes
weight_specs = [
    ('Flat / equal', 1.0, 1.0),
    ('Mildly front', 1.2, 3.0),
    ('Moderately front', 1.5, 5.0),
    ('Strongly front', 1.8, 8.0),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
K_sens = 12
j = np.arange(K_sens)

ax = axes[0]
for name, t1, t2 in weight_specs:
    w = beta_weights(K_sens, t1, t2)
    ax.plot(j, w, linewidth=2, marker='o', markersize=4,
            label=f'{name} (θ={t1},{t2})')
ax.axhline(1/K_sens, color='gray', linestyle='--', linewidth=1, label='Equal weight')
ax.set_xlabel('Lag j')
ax.set_ylabel('Weight')
ax.set_title('Weight Shapes: Flat to Front-Loaded')
ax.legend(fontsize=8)

# For each weight shape, compute the information loss from the ragged edge
# Info loss = fraction of total weight in the missing lags
ax2 = axes[1]
names, info_loss_h1, info_loss_h2 = [], [], []
for name, t1, t2 in weight_specs:
    w = beta_weights(K_sens, t1, t2)
    loss_h1 = w[0]               # Weight on the single missing lag (h=1)
    loss_h2 = w[0] + w[1]        # Weight on two missing lags (h=2)
    names.append(name)
    info_loss_h1.append(loss_h1)
    info_loss_h2.append(loss_h2)
    print(f"{name}: w[0]={w[0]:.4f}, w[0]+w[1]={w[0]+w[1]:.4f} "
          f"(vs equal {1/K_sens:.4f} and {2/K_sens:.4f})")

x_pos = np.arange(len(weight_specs))
width = 0.35
ax2.bar(x_pos - width/2, info_loss_h1, width, color='#ff7f0e', alpha=0.8,
        label='Weight on missing lag (h=1)')
ax2.bar(x_pos + width/2, info_loss_h2, width, color='#d62728', alpha=0.8,
        label='Weight on missing lags (h=2)')
ax2.axhline(1/K_sens,   color='gray', linestyle='--', linewidth=1, label='Equal weight for 1 lag')
ax2.axhline(2/K_sens,   color='silver', linestyle=':', linewidth=1, label='Equal weight for 2 lags')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([n.split()[0] for n in names], fontsize=9)
ax2.set_ylabel('Fraction of weight in missing lags')
ax2.set_title('Information Loss from Ragged Edge\nby Weight Shape')
ax2.legend(fontsize=8)

plt.suptitle('Ragged Edge Sensitivity to Weight Shape', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('ragged_edge_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Self-Check

In [ ]:
print("Self-check...")
passed = 0
total = 5

# 1. Backward shift at h=1 reduces K by 1
X_trunc = apply_ragged_edge(X_sim, 1, 'truncate')
assert X_trunc.shape[1] == K_SIM - 1, \
    f"FAIL 1: Expected K={K_SIM-1}, got {X_trunc.shape[1]}"
passed += 1
print(f"  PASS 1: Backward shift K: {K_SIM} -> {X_trunc.shape[1]}")

# 2. Re-weight keeps K but zeros the first h columns
X_rew = apply_ragged_edge(X_sim, 2, 'reweight')
assert X_rew.shape[1] == K_SIM, f"FAIL 2: Re-weight should keep K={K_SIM}"
assert np.all(X_rew[:, :2] == 0), "FAIL 2: First 2 columns should be zero for h=2"
passed += 1
print(f"  PASS 2: Re-weight preserves K={K_SIM}, zeros first 2 columns")

# 3. True weight on j=0 > 1/K for front-loaded Beta with theta2 > theta1
w_front = beta_weights(K_SIM, TRUE_THETA1, TRUE_THETA2)
assert w_front[0] > 1/K_SIM, \
    f"FAIL 3: w[0]={w_front[0]:.4f} should be > equal weight {1/K_SIM:.4f}"
passed += 1
print(f"  PASS 3: True w[0]={w_front[0]:.4f} > equal weight={1/K_SIM:.4f} (front-loaded)")

# 4. Complete-quarter estimate is less biased than 1-month vintage
# (in expectation; we test that h=0 beta_rmse <= h=2 beta_rmse)
if (0, 'truncate') in mc_summary and (2, 'truncate') in mc_summary:
    rmse0 = mc_summary[(0, 'truncate')]['beta_rmse']
    rmse2 = mc_summary[(2, 'truncate')]['beta_rmse']
    assert rmse0 <= rmse2 * 1.1, \
        f"FAIL 4: Complete-quarter RMSE ({rmse0:.4f}) should be <= 1-month RMSE ({rmse2:.4f})"
    passed += 1
    print(f"  PASS 4: Complete-quarter beta RMSE ({rmse0:.4f}) <= 1-month ({rmse2:.4f})")
else:
    print("  SKIP 4: MC results not available")

# 5. Info loss is larger for front-loaded weights than flat
w_flat = beta_weights(K_SIM, 1.0, 1.0)
w_front_check = beta_weights(K_SIM, 1.8, 8.0)
info_loss_flat  = w_flat[0]
info_loss_front = w_front_check[0]
assert info_loss_front > info_loss_flat, \
    f"FAIL 5: Front-loaded info loss ({info_loss_front:.4f}) should be > flat ({info_loss_flat:.4f})"
passed += 1
print(f"  PASS 5: Front-loaded info loss ({info_loss_front:.4f}) > flat ({info_loss_flat:.4f})")

print(f"\n{'='*40}")
print(f"Self-check: {passed}/{total} passed")
if passed == total:
    print("All checks passed.")

## Summary

| Finding | Implication |
|---------|-------------|
| Backward shift reduces K | Slight loss of information; simple to implement |
| Re-weighting preserves scale | More theoretically correct; similar practical performance |
| Front-loaded weights lose more information at ragged edge | BIC selection of theta matters for nowcast timing |
| MC bias is small for both strategies | Either approach acceptable with T≥80 |

### Next

**Notebook 03:** Forecast evolution — track the nowcast path across multiple quarters to evaluate systematic biases.